# CyberTwin Preprocessing and Feature Engineering (Milestone 2)

This notebook implements the data preprocessing and feature engineering pipeline for the CyberTwin project.

## 1. Load raw dataset
We begin by loading the raw network traffic dataset.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import joblib
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
import sys

plt.style.use('seaborn-v0_8-darkgrid')
df = pd.read_csv('../data/raw/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv')
df.columns = df.columns.str.strip()
print(f"Raw shape: {df.shape}")


## 2. Verify Milestone 1 findings
We confirm the class imbalance, presence of missing values, and identifying column structures as documented in `EDA_SUMMARY.md`.


In [ ]:
print(df['Label'].value_counts())
print(f"Total Missing: {df.isna().sum().sum()}")
print(f"Total Infinite: {np.isinf(df.select_dtypes(include=[np.number])).sum().sum()}")


## 3. Feature classification
We classify features into:
- Target (Label)
- Identifiers / Leakage risks (Destination Port, Source IP, etc.)
- Behavioral Traffic features (Flow Duration, Packets/s, etc.)


In [ ]:
# See DATASET_REPORT.md for full breakdown. 
target = 'Label'
identifiers = ['Destination Port', 'Flow ID', 'Source IP', 'Destination IP', 'Timestamp']


## 4. Leakage-control decisions
We must remove identifiers because a Digital Twin or a generalizable NIDS should not memorize IP addresses or specific ports. It should learn the *behavior* of the traffic.


In [ ]:
drop_cols = [c for c in identifiers if c in df.columns]
print(f"Dropping leakage features: {drop_cols}")
df = df.drop(columns=drop_cols)


## 5. Duplicate handling
Network flows can have exact duplicates. Removing them ensures the model doesn't overfit to repeated identical flows.


In [ ]:
n_dupes = df.duplicated().sum()
print(f"Duplicates found: {n_dupes}")
df = df.drop_duplicates().reset_index(drop=True)
print(f"Shape after duplicate removal: {df.shape}")


## 6. Missing/infinite value handling
We replace infinite values with NaN so they can be imputed properly later in the pipeline.


In [ ]:
df = df.replace([np.inf, -np.inf], np.nan)


## 7. Feature engineering
We create behavioral features to help the model learn ratios and totals. 


In [ ]:
df['Total_Packets'] = df['Total Fwd Packets'] + df['Total Backward Packets']
df['Fwd_Bwd_Packet_Ratio'] = df['Total Fwd Packets'] / (df['Total Backward Packets'] + 1e-6)
df['Fwd_Bwd_Byte_Ratio'] = df['Total Length of Fwd Packets'] / (df['Total Length of Bwd Packets'] + 1e-6)


## 8. Prepare Labels
We preserve multiclass labels and create a binary detection target.

In [ ]:
df['Label_multiclass'] = df['Label']
df['Label_binary'] = (df['Label'] != 'BENIGN').astype(int)


## 9. Drop Constant Features
Constant features provide no predictive power.

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns
numeric_cols = [c for c in numeric_cols if c not in ['Label_binary']]
constant_cols = [c for c in numeric_cols if df[c].nunique(dropna=True) <= 1]
print(f"Dropping constant cols: {constant_cols}")
df = df.drop(columns=constant_cols)


## 10. Train/validation/test split
We split 70/15/15 using stratification on the binary label to maintain class balance across splits.


In [ ]:
X = df.drop(columns=['Label', 'Label_multiclass', 'Label_binary'])
y = df[['Label_multiclass', 'Label_binary']]
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y['Label_binary'])
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp['Label_binary'])
print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")


## 11. Class distribution
Visualizing the class distribution across splits.


In [ ]:
def plot_dist(y_df, name):
    plt.figure(figsize=(6,4))
    sns.countplot(data=y_df, x='Label_binary')
    plt.title(f'Class Distribution - {name}')
    plt.savefig(f'../results/figures/class_distribution_{name.lower()}.png')
    plt.close()

plot_dist(y_train, 'Train')
plot_dist(y_val, 'Val')
plot_dist(y_test, 'Test')
print("Distributions plotted.")


## 12. Fit preprocessing pipeline
We use `SimpleImputer(median)` and `RobustScaler()`. RobustScaler is chosen because EDA revealed extreme outliers in network flow metrics (like duration).


In [ ]:
preprocessor = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())
])

# Fit ONLY on train
preprocessor.fit(X_train)


## 13. Transform datasets
Transform all splits using the fitted preprocessor.


In [ ]:
X_train_scaled = pd.DataFrame(preprocessor.transform(X_train), columns=X_train.columns)
X_val_scaled = pd.DataFrame(preprocessor.transform(X_val), columns=X_val.columns)
X_test_scaled = pd.DataFrame(preprocessor.transform(X_test), columns=X_test.columns)

# Feature distributions plot
plt.figure(figsize=(10,4))
sns.histplot(X_train['Flow Duration'].dropna(), bins=50)
plt.title('Flow Duration BEFORE scaling')
plt.savefig('../results/figures/feature_distribution_before_scaling.png')
plt.close()

plt.figure(figsize=(10,4))
sns.histplot(X_train_scaled['Flow Duration'].dropna(), bins=50)
plt.title('Flow Duration AFTER RobustScaler')
plt.savefig('../results/figures/feature_distribution_after_scaling.png')
plt.close()


## 14. Validate transformed data
Check for NaN or Inf.


In [ ]:
print(f"NaN in train: {X_train_scaled.isna().sum().sum()}")
print(f"Inf in train: {np.isinf(X_train_scaled).sum().sum()}")


## 15. Save processed datasets


In [ ]:
# Data is saved via the preprocessing.py script, this notebook is for interactive exploration.
print("Finished!")
